Importing the libraries

In [1]:
import pandas as pd

In [5]:
df=pd.read_csv(r'C:\Learning\Week-01(Python & OOP)\Learning-Curve\Week-5\Day-1\Regression\archive\train.csv',na_values=['NA','?'])
df['filename']="clips/clips-"+df['id'].astype(str)+".png"

Separate into training and testing data

In [6]:
train_pct=0.9
train_cut=int(len(df)*train_pct)

df_train=df[0:train_cut]
df_validate=df[train_cut:]

print(f"training size: {len(df_train)}")
print(f"validation size: {len(df_validate)}")

training size: 18000
validation size: 2000


In [7]:
df_train

,id,clip_count,filename
0,30001,11,clips-30001.png
1,30002,2,clips-30002.png
2,30003,26,clips-30003.png
3,30004,41,clips-30004.png
4,30005,49,clips-30005.png
...,...,...,...
17995,47996,9,clips-47996.png
17996,47997,45,clips-47997.png
17997,47998,52,clips-47998.png
17998,47999,63,clips-47999.png


Making the Model

In [18]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

df = pd.read_csv(
    r'C:\Learning\Week-01(Python & OOP)\Learning-Curve\Week-5\Day-1\Regression\archive\train.csv'
)

df['filename'] = "clips/clips-" + df['id'].astype(str) + ".png"

train_cut = int(len(df) * 0.9)
df_train = df[:train_cut]
df_validate = df[train_cut:]

Images_dir = r"C:\Learning\Week-01(Python & OOP)\Learning-Curve\Week-5\Day-1\Regression\archive\clips-data-2020"

training_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)

train_generator = training_datagen.flow_from_dataframe(
    dataframe=df_train,
    directory=Images_dir,
    x_col='filename',
    y_col='clip_count',
    target_size=(256, 256),
    batch_size=32,
    class_mode='other'
)

val_generator = validation_datagen.flow_from_dataframe(
    dataframe=df_validate,
    directory=Images_dir,
    x_col='filename',
    y_col='clip_count',
    target_size=(256, 256),
    batch_size=32,
    class_mode='other'
)


Found 18000 validated image filenames.
Found 2000 validated image filenames.


In [20]:
from tensorflow.keras.callbacks import EarlyStopping

model = tf.keras.models.Sequential([
    # Note the input shape is the desired size of the image 150x150 with 3 bytes color
    # This is the first convolution
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', input_shape=(256, 256, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    # The second convolution
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Flatten(),
    # 512 neuron hidden layer
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(1, activation='linear')
])


model.summary()
epoch_steps = 250 # needed for 2.2
validation_steps = len(df_validate)
model.compile(loss = 'mean_squared_error', optimizer='adam')
monitor = EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=5, verbose=1, mode='auto',
        restore_best_weights=True)
history = model.fit(train_generator,  
  verbose = 1, 
  validation_data=val_generator, callbacks=[monitor], epochs=1)
#  steps_per_epoch=epoch_steps, validation_steps=validation_steps, # needed for 2.2

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 254, 254, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 127, 127, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 125, 125, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 246016)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │   125,960,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,999,937 (480.65 MB)

 Trainable params: 125,999,937 (480.65 MB)

 Non-trainable params: 0 (0.00 B)

563/563 ━━━━━━━━━━━━━━━━━━━━ 877s 2s/step - loss: 172.4267 - val_loss: 14.0873
Restoring model weights from the end of the best epoch: 1.


In [29]:
df_test = pd.read_csv(
    r"C:\Learning\Week-01(Python & OOP)\Learning-Curve\Week-5\Day-1\Regression\archive\test.csv", 
    na_values=['NA', '?'])

df_test['filename']="clips/clips-"+df_test["id"].astype(str)+".png"
IMAGES_DIR = r"C:\Learning\Week-01(Python & OOP)\Learning-Curve\Week-5\Day-1\Regression\archive\clips-data-2020"


test_datagen = ImageDataGenerator(rescale = 1./255)

test_generator = test_datagen.flow_from_dataframe(
        dataframe=df_test,
        directory=IMAGES_DIR,
        x_col="filename",
        batch_size=1,
        shuffle=False,
        target_size=(256, 256),
        class_mode=None)

Found 5000 validated image filenames.


In [30]:
test_generator.reset()
print(test_generator.n)
pred = model.predict(test_generator)

5000
   3/5000 ━━━━━━━━━━━━━━━━━━━━ 2:20 28ms/step 

5000/5000 ━━━━━━━━━━━━━━━━━━━━ 156s 31ms/step


In [2]:
import pandas as pd
df_submit = pd.DataFrame({'id':df_test['id'],'clip_count':pred.flatten()})

NameError: name 'df_test' is not defined